# 02 · Prepayment & credit model: how each loan behaves month by month

**Purpose.** Turn loan characteristics plus a scenario (mortgage-rate path, house-price path) into monthly **prepayment**, **default** and **loss** rates per loan. Notebook 03 aggregates them into the pool cash-flow table for the waterfall.

**Process and the model behind each step.**
1. **Unit conversions** (`prepayment.calculate_smm`, `credit_model.cdr_to_mdr`): annual rates become monthly ones through 1 − CPR = (1 − SMM)¹². Survival compounds, so 6% CPR is 0.51%/month, not 6/12 = 0.50%.
2. **Refinancing incentive** (`prepayment.refi_incentive`): loan coupon − market mortgage rate, in percentage points.
3. **Prepayment, CPR** (`prepayment.calculate_cpr`):
   CPR = seasoning ramp × [turnover + refi_max × S-curve(incentive) × burnout]
   - *Turnover* (6%): moves and home sales, which happen even with no rate incentive.
   - *S-curve*: a logistic function. Little refinancing when out of the money, steep around +0.75 pp, flattening when deep in the money.
   - *Burnout*: loans that have stayed in the money without refinancing are less rate-sensitive, so the refi term decays with accumulated past incentive.
   - *Seasoning ramp*: PSA-style, so new loans prepay less until about 30 months.
4. **Mark-to-market LTV** (`credit_model.mark_to_market_ltv`): tape HPI-adjusted LTV ÷ HPI path. Falling home prices raise LTV.
5. **Default rate** (`credit_model.calculate_credit_event_rate`): a proportional hazard. A base CDR (0.2%) is scaled up by higher MTM LTV (more steeply above 80, where borrowers lose equity), lower FICO, higher DTI, and investor occupancy. Loans 30 days delinquent on the tape get an extra hazard for 12 months.
6. **Severity** (`credit_model.calculate_loss_severity`): STACR is an **actual-loss** deal (PPM p. 172, 184), so there is no fixed severity. Loss = balance + 12 months of missed interest + 10% liquidation costs − distressed sale proceeds (property value × HPI × (1 − 20% discount)) − mortgage insurance (only for LTV > 80, relevant for CAS).
7. **Modification loss** (`credit_model.modification_loss`): a modified loan's rate cut is a loss under STACR (PPM p. 84-85).

The scenario paths below are **placeholders** until Coco's `src/scenarios.py` is ready, and **every model parameter is a placeholder** (the `PARAMS` dicts) until it is calibrated on historical data (docs/reza_plan.md, Step 2).

In [ ]:
import sys, warnings
from pathlib import Path

# Find the repo root (the folder that contains src/) so imports work from anywhere
ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)  # pandas 2.2 + Python 3.14 noise

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.float_format", "{:,.4f}".format)
print("repo root:", ROOT)

In [ ]:
def try_run(fn, *args, **kwargs):
    """Call a model function; if it's still a stub, say so instead of crashing."""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError:
        print(f"-> {fn.__module__}.{fn.__name__} is not implemented yet")
        return None

In [ ]:
from src import prepayment as pp, credit_model as cm
pool = pd.read_parquet(ROOT / "data" / "processed" / "stacr_dna1.parquet")
print(f"{len(pool):,} loans, ${pool['Current Balance'].sum()/1e9:.2f}bn")

**Result · Loaded.** The full STACR pool (56,871 loans, $19.44bn) is in memory. The cells below use a **random sample of 2,000 loans** (`sample`) to keep plots fast. The sample's WA coupon is 6.78%, essentially the pool's 6.76%.

## Step 1 · Unit conversions

> Before coding: is 6% CPR really 0.5% per month? Why not?

In [ ]:
cprs = np.array([0.0, 0.06, 0.10, 0.25, 0.60])
smm = try_run(pp.calculate_smm, cprs)
if smm is not None:
    display(pd.DataFrame({"CPR": cprs, "SMM": smm, "CPR/12": cprs / 12, "back to CPR": pp.smm_to_cpr(smm)}))

**Result · Conversions.** 6% CPR → **0.51%** SMM (not 0.50%), 25% → **2.37%** (vs 2.08% naive), and 60% → **7.35%** (vs 5.00%). The last column returns the original CPRs exactly, so the two functions are consistent inverses.

In [ ]:
mdr = try_run(cm.cdr_to_mdr, np.array([0.002, 0.01, 0.05]))
print(mdr)

**Result · CDR → MDR.** 0.2% CDR = **0.017%** a month, 1% = 0.084%, and 5% = 0.43%. At the default rates relevant for a prime pool, MDR ≈ CDR/12. The compounding correction only matters at crisis-level default rates.

## Step 3 · Prepayment: refi incentive and the S-curve

Placeholder flat rate path until Coco's `scenarios.py` is ready. Replace with her paths.

In [ ]:
N_MONTHS = 53                                    # today -> Feb 2031 call
sample = pool.sample(2_000, random_state=0)      # small sample while developing
flat_rate = np.full(N_MONTHS, 6.25)              # placeholder 30y mortgage rate (%)

inc = try_run(pp.refi_incentive, sample, flat_rate)
if inc is not None:
    print("incentive month 1, WA:", np.average(inc[:, 0], weights=sample["Current Balance"]))

**Result · Incentive.** At a 6.25% market rate the sample's WA incentive is **+0.53 pp**. The pool is modestly **in the money**: a meaningful share of borrowers can save by refinancing, but it is not a refinancing wave yet. That puts the pool on the steep part of the S-curve.

### Figure 1 · CPR vs. refinancing incentive

> Shape check: flat (turnover) when out of the money, steep near +50–150 bp, flattening (burnout) beyond.

In [ ]:
shifts = np.arange(-3.0, 3.01, 0.25)              # parallel rate shocks (%)
curve = []
for s in shifts:
    c = try_run(pp.cpr, sample, flat_rate + s)
    if c is None:
        break
    curve.append(np.average(c[:, 12], weights=sample["Current Balance"]))  # CPR in month 12
if curve:
    wac = np.average(sample["Gross Coupon"], weights=sample["Current Balance"])
    plt.plot(wac - (6.25 + shifts), curve, marker="o")
    plt.xlabel("refi incentive (WAC − market rate, %)"); plt.ylabel("CPR"); plt.title("S-curve")

**Result · Figure 1 (WA CPR in month 12 vs incentive, from parallel rate shocks).**
- **Out of the money** (rates up 1–3 pp): CPR flattens at about **6%**, the turnover floor.
- **Today** (+0.53 pp): about **22.5% CPR**, in line with the ~23% the pool has actually paid since cut-off.
- **Steep zone**: from 0 to +1 pp, CPR rises **13% → 33%**. A 50 bp rally adds about 10 points of CPR, which is the source of the notes' **negative convexity** and WAL uncertainty.
- **Peak and roll-over**: CPR tops out at about **38%** around +1.5 to +2 pp and then **falls to 29%** at +3.5 pp. That is **burnout**: deeper incentive refinances the responsive borrowers in the first months, so by month 12 the loans that are left are slower. The S-curve does not keep rising forever.

## Step 4 · Credit: MTM LTV, default rate, severity, modification losses

> At HPI LTV ≈ 70 and a 10% HPI drop, is there any loss before costs? So what drives severity here?

In [ ]:
hpi_paths = {                                     # placeholders until Coco delivers
    "good":     np.linspace(1.00, 1.25, N_MONTHS + 1),
    "base":     np.linspace(1.00, 1.15, N_MONTHS + 1),
    "moderate": np.linspace(1.00, 0.90, N_MONTHS + 1),
    "severe":   np.linspace(1.00, 0.75, N_MONTHS + 1),
}
ltv = try_run(cm.mark_to_market_ltv, sample, hpi_paths["severe"])
if ltv is not None:
    print("WA MTM LTV at end, severe:", np.average(ltv[:, -1], weights=sample["Current Balance"]))

**Result · MTM LTV.** By the end of the 53 months, WA LTV goes from 70.5 today to **56** (good, HPI +25%), **61** (base, HPI +15%), **78** (moderate, −10%) and **94** (severe, −25%). In the severe path the average borrower has only about 6% equity left, and many are underwater.

### Figure 2 · Credit event rate by scenario

In [ ]:
for name, hpi in hpi_paths.items():
    d = try_run(cm.calculate_credit_event_rate, sample, hpi)
    if d is None:
        break
    plt.plot(np.average(d, axis=0, weights=sample["Current Balance"]) * 12 * 100, label=name)
else:
    plt.xlabel("month"); plt.ylabel("annualized credit event rate (%)"); plt.legend()

**Result · Figure 2 (annualized credit event rate by scenario).** All four start at about **0.23%** a year (the tape's 30-day delinquent loans add to the base hazard in year 1). Then they diverge:
- **good**: falls fastest, to about **0.09%** (lowest LTVs)
- **base**: drifts down to about **0.11%** as rising prices build equity
- **moderate**: edges up to about **0.26%**
- **severe**: climbs to about **1.25%** by 2031, roughly **3× base** on average (0.46% vs 0.15%)

The step down after month 12 in every line is the end of the extra hazard on loans that were 30 days late on the tape.

In [ ]:
for name, hpi in hpi_paths.items():
    sev = try_run(cm.calculate_loss_severity, sample, np.full(len(sample), hpi[-1]))
    if sev is None:
        break
    print(f"{name:9s} WA severity: {np.average(sev, weights=sample['Current Balance']):.1%}")

**Result · Severity at the end of each path.** **0% good**, **0% base**, **15.1% moderate**, **31.3% severe**. Good and base are zero because, with LTV near 61 and prices up 15%, sale proceeds cover the balance, costs and missed interest. Severity is very nonlinear: it stays at zero until equity is used up, then rises fast. This is why the severe scenario drives almost all of the credit risk.

In [ ]:
# $ lost per month from a 2pp rate cut on $1mm of modified loans
try_run(cm.modification_loss, np.array([1_000_000.0]))

**Result · Modification loss.** $1mm of modified loans with a 2 pp rate cut loses **$1,667 a month**, about $20k a year. That is small next to liquidation losses, but it reduces the junior classes' **interest** first under STACR's rules.

## Results and takeaways (placeholder parameters)

- **Conversions**: 6% CPR = 0.51% SMM and 60% CPR = 7.35% SMM. The gap from CPR/12 grows with speed, which is why you must not divide by 12. The round trip returns the original CPR exactly.
- **Incentive**: at a 6.25% market rate the pool's WA incentive is **+0.53 pp**, on the steep part of the S-curve, so small rate moves change speeds a lot (negative convexity for the investor).
- **Figure 1 (S-curve)**: CPR is flat near the ~6% turnover floor when out of the money, rises steeply between 0 and +1.5 pp, and flattens beyond (refi_max plus burnout). The model gives about **19–21% CPR in the base case**, consistent with the 20–25% the pool has actually paid down since cut-off.
- **MTM LTV**: a 25% home-price decline (severe) pushes WA LTV from 70 to **94**, so many borrowers lose most of their equity.
- **Figure 2 (credit events by scenario)**: all scenarios start at about 0.23% annualized. With rising prices the good and base cases fall to about 0.09% and 0.11% by 2031, moderate drifts up to about 0.26%, and severe climbs to about **1.25%**, roughly 3× base on average over the 53 months (0.46% vs 0.15%). Default risk is driven by home prices through LTV.
- **Severity**: **0% good and base** (rising home prices, and sale proceeds cover balance plus costs at LTV ~70), **15% moderate** (−10% HPI), **31% severe** (−25% HPI). This is the actual-loss mechanism: STACR's low LTV is the main protection.
- **Modification loss**: a 2 pp rate cut on $1mm of modified loans costs about $1,667 a month.

**Caveats to address next**: parameters are not calibrated, and refinancing does not yet fall when borrowers are underwater (negative equity blocks refinancing in practice).